# 08E_Longitudinal_History_Rebuilder

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

c:\Users\Saanvi\anaconda3\envs\COURSE\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
survey_files = {2011:'../Raw Data/StackOverflow/survey_2011.csv',2012:'../Raw Data/StackOverflow/survey_2012.csv',2013:'../Raw Data/StackOverflow/survey_2013.csv',2014:'../Raw Data/StackOverflow/survey_2014.csv',2015:'../Raw Data/StackOverflow/survey_2015.csv',2016:'../Raw Data/StackOverflow/survey_2016.csv',2017:'../Raw Data/StackOverflow/survey_2017.csv',2018:'../Raw Data/StackOverflow/survey_2018.csv',2019:'../Raw Data/StackOverflow/survey_2019.csv',2020:'../Raw Data/StackOverflow/survey_2020.csv',2021:'../Raw Data/StackOverflow/survey_2021.csv',2022:'../Raw Data/StackOverflow/survey_2022.csv',2023:'../Raw Data/StackOverflow/survey_2023.csv',2024:'../Raw Data/StackOverflow/survey_2024.csv',2025:'../Raw Data/StackOverflow/survey_2025.csv'}

In [3]:
target_columns=['Which languages are you proficient in?','Which of the following languages or technologies have you used significantly in the past year?','tech_do','HaveWorkedLanguage','HaveWorkedFramework','HaveWorkedDatabase','HaveWorkedPlatform','LanguageWorkedWith','DatabaseWorkedWith','PlatformWorkedWith','FrameworkWorkedWith','WebFrameWorkedWith','WebframeWorkedWith','MiscTechWorkedWith','LanguageHaveWorkedWith','DatabaseHaveWorkedWith','PlatformHaveWorkedWith','WebframeHaveWorkedWith','MiscTechHaveWorkedWith','ToolsTechHaveWorkedWith']

In [4]:
def read_survey(path):
    for enc in [None,'utf-8','latin1','cp1252']:
        try:
            return pd.read_csv(path, low_memory=False) if enc is None else pd.read_csv(path, encoding=enc, low_memory=False)
        except:
            pass
    raise ValueError(path)

In [5]:
def extract_skill_counts(df, cols):
    counter=Counter()
    available=[c for c in cols if c in df.columns]
    for col in available:
        for row in df[col].dropna().astype(str):
            row=row.replace(',', ';')
            parts=[x.strip().lower() for x in row.split(';') if len(x.strip())>1]
            counter.update(parts)
    return counter

In [6]:
all_records=[]
for year,path in tqdm(survey_files.items()):
    try:
        df=read_survey(path)
        counts=extract_skill_counts(df,target_columns)
        for skill,demand in counts.items():
            all_records.append({'skill':skill,'year':year,'demand':demand})
        print(year,len(counts))
    except Exception as e:
        print(year,e)

 13%|█▎        | 2/15 [00:00<00:00, 19.55it/s]

2011 1
2012 1
2013 0


 27%|██▋       | 4/15 [00:00<00:00, 14.01it/s]

2014 0
2015 0


 40%|████      | 6/15 [00:01<00:03,  2.83it/s]

...

2023 230


 93%|█████████▎| 14/15 [00:17<00:01,  1.96s/it]

2024 218


100%|██████████| 15/15 [00:19<00:00,  1.32s/it]

2025 139


In [7]:
history_v2=pd.DataFrame(all_records)
history_v2['skill']=history_v2['skill'].astype(str).str.lower().str.strip()
history_v2=history_v2[history_v2['skill'].str.len()>1]
history_v2=history_v2.drop_duplicates(subset=['skill','year'])
print(history_v2.shape)
history_v2.head()

(1201, 3)


,skill,year,demand
0,java,2011,862
1,java,2012,2349
2,ios,2016,4498
3,objective-c,2016,3202
4,android,2016,8601


In [8]:
coverage=history_v2.groupby('skill')['year'].nunique()
print(coverage.describe())

count    335.000000
mean       3.585075
std        2.740976
min        1.000000
25%        1.000000
50%        3.000000
75%        5.000000
max       12.000000
Name: year, dtype: float64


In [9]:
history_v2.to_csv('../Generated Datasets/skill_demand_history_v2.csv',index=False)
print('Saved Successfully')

Saved Successfully


In [10]:
coverage = (
    history_v2
    .groupby('skill')['year']
    .nunique()
    .sort_values(ascending=False)
)

coverage.head(50)

skill
java               12
c++                10
c#                 10
swift              10
cassandra          10
ruby               10
sql                10
scala              10
go                 10
php                10
python             10
redis              10
node.js            10
javascript         10
mongodb            10
rust               10
sqlite              9
f#                  9
firebase            9
oracle              9
cordova             9
assembly            9
typescript          9
dart                9
perl                9
mysql               9
postgresql          9
hadoop              9
vba                 9
objective-c         9
django              8
matlab              8
haskell             8
mariadb             8
microsoft azure     8
heroku              8
kotlin              8
drupal              8
erlang              8
clojure             8
elasticsearch       8
wordpress           8
xamarin             8
torch/pytorch       7
tensorflow          7
asp.

In [13]:
alias_map = {

    'torch/pytorch': 'pytorch',

    'ms sql server': 'microsoft sql server',
    'sql server': 'microsoft sql server',

    'google cloud platform': 'google cloud',
    'gcp': 'google cloud',

    'amazon web services': 'aws',

    'azure': 'microsoft azure',

    'react.js': 'react',
    'vue': 'vue.js',

    'nodejs': 'node.js',
    'node': 'node.js',

    'postgres': 'postgresql',

    'js': 'javascript',

    'tf': 'tensorflow'
}

history_v2['skill'] = (history_v2['skill']
    .str.lower()
    .str.strip()
    .replace(alias_map)
)

In [16]:
coverage = (
    history_v2
    .groupby('skill')['year']
    .nunique()
    .sort_values(ascending=False)
)

coverage.head(50)


skill
java                    12
swift                   10
c#                      10
cassandra               10
sql                     10
go                      10
php                     10
node.js                 10
mongodb                 10
microsoft sql server    10
javascript              10
microsoft azure         10
scala                   10
rust                    10
ruby                    10
python                  10
redis                   10
c++                     10
sqlite                   9
typescript               9
dart                     9
f#                       9
firebase                 9
cordova                  9
vba                      9
assembly                 9
hadoop                   9
objective-c              9
oracle                   9
mysql                    9
postgresql               9
react                    9
perl                     9
wordpress                8
drupal                   8
erlang                   8
clojure               

In [15]:
display(coverage)

,skill,history_length
0,.net,3
1,.net (5+),2
2,.net core,4
3,.net core / .net 5,1
4,.net framework,1
...,...,...
326,yandex cloud,1
327,yarn,5
328,yii 2,1
329,zephyr,1


In [17]:
history_v2.to_csv('../Generated Datasets/skill_demand_history_v2.csv',index=False)
print('Saved Successfully')

Saved Successfully
